In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import *

## SCD -2 Implementation

In [0]:
from pyspark.sql import Row
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType,
    BooleanType,
    DateType
)

target_data = [
    Row(id=1, name="John", city="US",  start_date="2024-01-01", end_date=None, is_current=True),
    Row(id=2, name="Mike", city="IN",  start_date="2024-01-01", end_date=None, is_current=True),
    Row(id=3, name="Sara", city="UK",  start_date="2024-01-01", end_date=None, is_current=True),
]

schema = StructType([
    StructField("id", IntegerType(), False),
    StructField("name", StringType(), False),
    StructField("city", StringType(), False),
    StructField("start_date", StringType(), False),
    StructField("end_date", StringType(), True),
    StructField("is_current", BooleanType(), False)
])

df_tgt = spark.createDataFrame(target_data, schema)

df_tgt.write.mode("overwrite").format("delta").saveAsTable("catalog_1.schema_1.dim_customer")

print("Target (Dimension) Table Loaded")
display(df_tgt)

In [0]:
from pyspark.sql import Row
from pyspark.sql.functions import current_date, lit

source_data = [
    Row(id=1, name="John", city="IND"),     #city change (US -> IND)
    Row(id=2, name="Mike", city="US"),     # city changed (IN -> UK)
    Row(id=3, name="Sara", city="UK"),     # no change
    Row(id=4, name="David", city="UK"),    # new record
    Row(id=5, name="Travis", city="IND"),
]

df_src = spark.createDataFrame(source_data)

# Add SCD2 default fields
df_src2 = (
    df_src.withColumn("start_date", current_date())
          .withColumn("end_date", lit(None))
          .withColumn("is_current", lit(True))
)

df_src2.show()

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import current_date

deltaTable = DeltaTable.forName(spark, "catalog_1.schema_1.dim_customer")

# Merge for SCD2
(
    deltaTable.alias("t")
    .merge(
        df_src2.alias("s"),
        "t.id = s.id AND t.is_current = True"
    )
    # When matched AND data has changed → expire the existing record
    .whenMatchedUpdate(
        condition="t.name <> s.name OR t.city <> s.city",
        set={
            "end_date": current_date(),
            "is_current": "False"
        }
    )
    # Insert new record for changed IDs
    .whenNotMatchedInsert(
        values={
            "id": "s.id",
            "name": "s.name",
            "city": "s.city",
            "start_date": "s.start_date",
            "end_date": "s.end_date",
            "is_current": "s.is_current"
        }
    ).execute()
)

(
    deltaTable.alias("t")
    .merge(
        df_src2.alias("s"),
        "t.id = s.id AND t.name = s.name AND t.city = s.city AND t.is_current = True"
    )
    .whenNotMatchedInsert(
        values={
            "id": "s.id",
            "name": "s.name",
            "city": "s.city",
            "start_date": "s.start_date",
            "end_date": "s.end_date",
            "is_current": "s.is_current"
        }
    ).execute()
)

In [0]:
%sql
select * from catalog_1.schema_1.dim_customer;
-- drop table catalog_1.schema_1.dim_customer;

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import current_date, lit

deltaTable = DeltaTable.forName(spark, "catalog_1.schema_1.dim_customer")

(
    deltaTable.alias("t")
    .merge(
        df_src2.alias("s"),
        "t.id = s.id AND t.is_current = true"
    )
    
    # Expire changed records
    .whenMatchedUpdate(
        condition="t.name <> s.name OR t.city <> s.city",
        set={
            "end_date": current_date(),
            "is_current": "False"
        }
    )
    
    # Insert new records (new IDs OR changed IDs)
    .whenNotMatchedInsert(
        values={
            "id": "s.id",
            "name": "s.name",
            "city": "s.city",
            "start_date": current_date(),
            "end_date": "null",
            "is_current": "True"
        }
    )
    
    .execute()
)

## Hashed Based SCD-2 Implementation

In [0]:
from pyspark.sql import Row
from pyspark.sql.functions import (
    col, current_timestamp, lit,
    sha2, concat_ws
)
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType,
    BooleanType,
    DateType
)

target_data = [
    Row(id=1, name="John", city="US",  start_date="2024-01-01", end_date=None, is_current=True),
    Row(id=2, name="Mike", city="IN",  start_date="2024-01-01", end_date=None, is_current=True),
    Row(id=3, name="Sara", city="UK",  start_date="2024-01-01", end_date=None, is_current=True),
]

schema = StructType([
    StructField("id", IntegerType(), False),
    StructField("name", StringType(), False),
    StructField("city", StringType(), False),
    StructField("start_date", StringType(), False),
    StructField("end_date", StringType(), True),
    StructField("is_current", BooleanType(), False)

])



df_tgt = spark.createDataFrame(target_data, schema)

df_tgt = df_tgt.withColumn(
    "hash_value",
    sha2(concat_ws("||",col("id"),col("name"), col("city")), 256)
)
display(df_tgt)
df_tgt.write.mode("overwrite").option("mergeSchema","true").format("delta").saveAsTable("catalog_1.schema_1.dim_customer_hash")

print("Target (Dimension) Table Loaded")


In [0]:
from pyspark.sql import Row
from pyspark.sql.functions import current_date, lit

source_data = [
    Row(id=1, name="John", city="IND"),     #city change (US -> IND)
    Row(id=2, name="Mike", city="US"),     # city changed (IN -> UK)
    Row(id=3, name="Sara", city="UK"),     # no change
    Row(id=4, name="David", city="IND"),    # new record
    Row(id=5, name="Travis", city="CAN"),
]

df_src = spark.createDataFrame(source_data)

# Add SCD2 default fields
# df_src2 = (
#     df_src.withColumn("start_date", current_date())
#           .withColumn("end_date", lit(None))
#           .withColumn("is_current", lit(True))
# )

# df_src2.show()

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import (
    col, current_timestamp, lit,
    sha2, concat_ws
)

#  Load Target Table
delta_table = DeltaTable.forName(spark, "catalog_1.schema_1.dim_customer_hash")

#  Remove duplicate source records (latest wins logic optional)
df_src = df_src2.dropDuplicates(["id"])

#  Generate hash for change detection
df_src = df_src.withColumn(
    "hash_value",
    sha2(concat_ws("||",col("id"),col("name"), col("city")), 256)
)

#  Add SCD metadata columns
df_src = (
    df_src
    .withColumn("start_date", current_timestamp())
    .withColumn("end_date", lit(None).cast("timestamp"))
    .withColumn("is_current", lit(True))
    # .withColumn("created_ts", current_timestamp())
    # .withColumn("updated_ts", current_timestamp())
)

#  Perform Single Atomic MERGE
(
    delta_table.alias("t")
    .merge(
        df_src.alias("s"),
        "t.id = s.id AND t.is_current = true"
    )

    # 5A️⃣ Expire changed records
    .whenMatchedUpdate(
        condition="t.hash_value <> s.hash_value",
        set={
            "end_date": current_timestamp(),
            "is_current": lit(False),
            # "updated_ts": current_timestamp()
        }
    )

    # 5B️⃣ Insert new or changed records
    .whenNotMatchedInsert(
        values={
            "t.id": "s.id",
            "t.name": "s.name",
            "t.city": "s.city",
            "t.hash_value": "s.hash_value",
            "t.start_date": "s.start_date",
            "t.end_date": "s.end_date",
            "t.is_current": "s.is_current"
            # "t.created_ts": "s.created_ts",
            # "t.updated_ts": "s.updated_ts"
        }
    )

    .execute()
)

# Second MERGE statement (fix indentation)
(
    delta_table.alias("t")
    .merge(
        df_src.alias("s"),
        "t.hash_value = s.hash_value AND t.is_current = True"
    )
    .whenNotMatchedInsert(
        values={
            "t.id": "s.id",
            "t.name": "s.name",
            "t.city": "s.city",
            "t.hash_value": "s.hash_value",
            "t.start_date": "s.start_date",
            "t.end_date": "s.end_date",
            "t.is_current": "s.is_current"
        }
    ).execute()
)

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import (
    col, sha2, concat_ws, current_timestamp, lit
)

target_table = DeltaTable.forName(spark, "catalog_1.schema_1.dim_customer_hash")

df_src1 = df_src.dropDuplicates(["id"])


df_src = (
    df_src1
    .withColumn("hash_value",
    # sha2(concat_ws("||",col("id"), col("name"), col("city")), 256))
    sha2(concat_ws("||",*[col(x) for x in df_src.columns]), 256))
    .withColumn("start_date", current_date())
    .withColumn("end_date", lit(None).cast("date"))
    .withColumn("is_current", lit(True))
)
display(df_src)
df_target = spark.table("catalog_1.schema_1.dim_customer_hash") \
                 .filter("is_current = true")
matched_rec_df = df_src.alias("s").join(df_target.alias("t"), "id", "left").select("s.*").withColumn("merge_key",lit(None)).filter("s.hash_value != t.hash_value")

# display(matched_rec_df)

df_src = df_src.withColumn("merge_key", col("id"))

df_stage = df_src.unionByName(matched_rec_df, allowMissingColumns=True)
display(df_stage)

(
    target_table.alias("t")
    .merge(
        df_stage.alias("s"),
        "t.id = s.merge_key"
    )
    
    .whenMatchedUpdate(
        condition="t.is_current = true and t.hash_value != s.hash_value",
        set={
            "end_date": current_date(),
            "is_current": lit(False)
        }
    )
    .whenNotMatchedInsert(
        values={
            "t.id": "s.id",
            "t.name": "s.name",
            "t.city": "s.city",
            "t.hash_value": "s.hash_value",
            "t.start_date": "s.start_date",
            "t.end_date": "s.end_date",
            "t.is_current": "s.is_current"
        }

    )    
    .execute()
)

In [0]:
%sql
select * from catalog_1.schema_1.dim_customer_hash

## SCD 2 Without adding hash column to source

In [0]:
from pyspark.sql import Row
from pyspark.sql.functions import (
    col, current_timestamp, lit,
    sha2, concat_ws
)
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType,
    BooleanType,
    DateType
)

target_data = [
    Row(id=1, name="John", city="US",  start_date="2024-01-01", end_date=None, is_current=True),
    Row(id=2, name="Mike", city="IN",  start_date="2024-01-01", end_date=None, is_current=True),
    Row(id=3, name="Sara", city="UK",  start_date="2024-01-01", end_date=None, is_current=True),
]

schema = StructType([
    StructField("id", IntegerType(), False),
    StructField("name", StringType(), False),
    StructField("city", StringType(), False),
    StructField("start_date", StringType(), False),
    StructField("end_date", StringType(), True),
    StructField("is_current", BooleanType(), False)

])



df_tgt = spark.createDataFrame(target_data, schema)

display(df_tgt)
df_tgt.write.mode("overwrite").option("mergeSchema","true").format("delta").saveAsTable("catalog_1.schema_1.dim_customer_hash_1")

print("Target (Dimension) Table Loaded")


In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import (
    col, sha2, concat_ws, current_timestamp, lit
)

target_table = DeltaTable.forName(spark, "catalog_1.schema_1.dim_customer_hash_1")

df_src = df_src2.dropDuplicates(["id"])


df_src = (
    df_src
    .withColumn("hash_value",
    sha2(concat_ws("||", col("name"), col("city")), 256))
    .withColumn("start_date", current_timestamp())
    .withColumn("end_date", lit(None).cast("timestamp"))
    .withColumn("is_current", lit(True))
)

df_src.display()

df_target = spark.table("catalog_1.schema_1.dim_customer_hash_1").filter("is_current = true")
display(df_target)

df_target = df_target.withColumn("hash_value",
    sha2(concat_ws("||", col("name"), col("city")), 256))

# df_target.display()

matched_rec_df = df_src.alias("s").join(df_target.alias("t"), "id", "left").select("s.*").withColumn("merge_key",lit(None)).filter("s.hash_value != t.hash_value")

display(matched_rec_df)

df_src = df_src.withColumn("merge_key", col("id"))

df_stage = df_src.unionByName(matched_rec_df, allowMissingColumns=True)
display(df_stage)

(
    target_table.alias("t")
    .merge(
        df_stage.alias("s"),
        "t.id = s.merge_key"
    )
    
    .whenMatchedUpdate(
        condition="t.is_current = true",
        set={
            "end_date": current_date(),
            "is_current": lit(False)
        }
    )
    .whenNotMatchedInsert(
        values={
            "t.id": "s.id",
            "t.name": "s.name",
            "t.city": "s.city",
            "t.start_date": "s.start_date",
            "t.end_date": "s.end_date",
            "t.is_current": "s.is_current"
        }

    )    
    .execute()
)

In [0]:
%sql
select * from catalog_1.schema_1.dim_customer_hash_1

## JSON Explode

In [0]:
json_df = spark.read.format("json").option("mode","dropCorrupt").option("multiline", "true").load("/Volumes/catalog_1/schema_1/vol_1/nested_json_data.json")

In [0]:
# json_df2 = spark.read.format("json").option("mode", "permissive").option("multiline", "true").load("/Volumes/catalog_1/schema_1/vol_1/nested_json_data.json")

In [0]:
json_df.printSchema()

In [0]:
json_df.select("data.year1.country1").display()

In [0]:
from pyspark.sql.functions import col, explode_outer

country_1_df = json_df.select(
    
    explode_outer(col("data.year1.country1.children")).alias("country_1_child"),
    explode_outer(col("data.year1.country2.children")).alias("country_2_child")
)
display(country_1_df)

In [0]:
from pyspark.sql.functions import col, explode_outer

country_2_df = json_df.withColumn(
    "country_2_child",
    explode_outer(col("data.year1.country2.children"))
).withColumn("Country", lit("Country 2"))
display(country_2_df)

In [0]:
json_df.display()

## Salting in the DF

In [0]:
from pyspark.sql.functions import col

# Large skewed table
employees_df = df

# Small table
dept_data = [
    ("IT", "Bangalore"),
    ("HR", "Mumbai"),
    ("Finance", "Delhi")
]

dept_df = spark.createDataFrame(dept_data, ["department", "location"])

In [0]:
from pyspark.sql.functions import floor, rand

salt_buckets = 5

employees_salted = employees_df.withColumn(
    "salt",
    floor(rand() * salt_buckets)
)

In [0]:
from pyspark.sql.functions import explode, array

dept_salted = dept_df.withColumn(
    "salt",
    explode(array([lit(i) for i in range(salt_buckets)]))
)

In [0]:
result = employees_salted.join(
    dept_salted,
    ["department", "salt"]
).drop("salt")

result.display()

In [0]:
customers_data = [
    (1, "Amit", "India"),
    (2, "Rahul", "USA"),
    (3, "Neha", "India"),
    (4, "John", "USA"),
    (5, "Sara", "UK")
]

customers = spark.createDataFrame(customers_data,
                                   ["customer_id", "customer_name", "country"])

In [0]:
orders_data = [
    (101, 1, "2024-01-01", 500),
    (102, 1, "2024-02-01", 700),
    (103, 2, "2024-01-10", 300),
    (104, 3, "2024-03-05", 1000),
    (105, 3, "2024-03-20", 1500),
    (106, 6, "2024-04-01", 800)  # invalid customer
]

orders = spark.createDataFrame(orders_data,
                                ["order_id", "customer_id", "order_date", "amount"])

In [0]:
payments_data = [
    (101, "Completed"),
    (102, "Failed"),
    (103, "Completed"),
    (104, "Completed"),
    (107, "Completed")  # invalid order
]

payments = spark.createDataFrame(payments_data,
                                  ["order_id", "payment_status"])

In [0]:
df_joined = customers.join(orders, "customer_id","full_outer").join(payments,"order_id","full_outer")

In [0]:
display(df_joined)

### Find Customes Who Didn't Place Any Order

In [0]:
df_joined.filter(col("order_id").isNull()).display()

### Find Customers Who Placed Orders But Payment Failed

In [0]:
df_joined.filter(col("payment_status") != "Failed").display()

In [0]:
# Generating Maximum Revenu from Which country

df_joined.groupBy("country").agg(sum(col("amount")))

### Scenario 
**Imagine you have a dataset of user clicks. A "session" is defined as a series of clicks where each click occurs within 30 minutes of the previous one. If more than 30 minutes pass, a new session starts.**
**Assign a unique session_id to each group of clicks for each user.**

- step 1 : read the dataset
- step 2 : we have to create a window for this for the user and group by for the group(window_function) 
- step 3: we have to assign a ID to the partitiond group (session_id) and check the dif of the id within the id (30 minutes differece)
- step 4 : load the data ( to capture or store the records)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_data = [
    ("User_A", 1000), ("User_A", 1500), ("User_A", 4000),
    ("User_B", 2000), ("User_B", 2100)
]

df = spark.createDataFrame(df_data, ["user_id", "click_time"])

# window by user ordered by click time
w = Window.partitionBy("user_id").orderBy("click_time")

# previous click time
df1 = df.withColumn(
    "prev_click_time",
    F.lag("click_time").over(w)
)

# time difference
df2 = df1.withColumn(
    "time_diff",
    F.col("click_time") - F.col("prev_click_time")
)

# new session flag
df3 = df2.withColumn(
    "new_session",
    F.when((F.col("time_diff") > 1800) | F.col("time_diff").isNull(), 1).otherwise(0)
)

# session id using cumulative sum
df_final = df3.withColumn(
    "session_id",
    F.sum("new_session").over(w)
)

df_final.show()